# 14 fonctions Pandas pour le Data Cleaning

Notebook de l'article [14 fonctions Pandas que les Data Scientists utilisent 60 % du temps pour le Data Cleaning](https://moncoachdata.com/blog/14-fonctions-pandas-pour-data-science/) (MonCoachData).

On nettoie et on transforme les matchs des Coupes du Monde de football (1930-2018) scrapés sur Wikipédia. Exécute les cellules **dans l'ordre** : les trois fichiers CSV sont lus directement depuis le dépôt GitHub `moncoachdata/blog-data-science` (il faut une connexion internet).

Si tu travailles sur ton ordinateur, installe d'abord Pandas (déjà présent dans Google Colab) : `pip install pandas`.

## 1. Charger les trois jeux de données avec `read_csv`

Attendu : `(457, 4) (64, 4) (508, 4)`.

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/14-fonctions-pandas-pour-data-science/"

df_historical_data = pd.read_csv(BASE + "fifa_worldcup_matches.csv")
df_fixture = pd.read_csv(BASE + "fifa_worldcup_fixture.csv")
df_missing_data = pd.read_csv(BASE + "fifa_worldcup_missing_data.csv")

In [ ]:
print(df_historical_data.shape, df_fixture.shape, df_missing_data.shape)

## 2. Supprimer les espaces (insécables) avec `str.strip`

Les noms d'équipes se terminent par `\xa0`, une espace insécable invisible.

In [ ]:
df_fixture.head(8)

In [ ]:
df_fixture["home"].head(3).tolist()

In [ ]:
df_fixture["home"] = df_fixture["home"].str.strip()
df_fixture["away"] = df_fixture["away"].str.strip()

In [ ]:
df_fixture["home"].head(3).tolist()

## 3. Valeurs manquantes : `isnull` et `dropna`

Attendu : 64 lignes vides, puis `(444, 4)` après `dropna`.

In [ ]:
df_missing_data.isnull().sum()

In [ ]:
df_missing_data[df_missing_data["home"].isnull()].head(5)

In [ ]:
df_missing_data = df_missing_data.dropna()
df_missing_data.shape

## 4. Fusionner avec `concat` et supprimer les doublons avec `drop_duplicates`

Attendu : `(901, 4)` et 0 doublon.

In [ ]:
df_historical_data = pd.concat([df_historical_data, df_missing_data], ignore_index=True)
df_historical_data.shape

In [ ]:
print(df_historical_data.duplicated().sum())

In [ ]:
df_historical_data = (
    df_historical_data
    .drop_duplicates()
    .sort_values("year", kind="stable")
    .reset_index(drop=True)
)
df_historical_data.shape

In [ ]:
df_historical_data.tail(6)

## 5. Repérer et supprimer une anomalie : `str.contains` et `drop`

Le match Suède-Autriche de 1938 est un forfait (`w/o[a]`). Attendu après suppression : `(900, 4)`.

In [ ]:
df_historical_data[df_historical_data["score"].str.contains("w/o")]

In [ ]:
walkover = df_historical_data[
    df_historical_data["home"].str.contains("Sweden")
    & df_historical_data["away"].str.contains("Austria")
].index
df_historical_data = df_historical_data.drop(index=walkover)
df_historical_data.shape

## 6. Corriger les incohérences avec une regex et `str.replace`

Attendu : 68 scores avec `(a.e.t.)`, puis 0 incohérence.

In [ ]:
incoherences = df_historical_data[df_historical_data["score"].str.contains(r"[^\d–]")]
len(incoherences)

In [ ]:
incoherences.head(6)

In [ ]:
df_historical_data["score"] = df_historical_data["score"].str.replace(r"[^\d–]", "", regex=True)
print(df_historical_data["score"].str.contains(r"[^\d–]").sum())

In [ ]:
df_historical_data["home"] = df_historical_data["home"].str.strip()
df_historical_data["away"] = df_historical_data["away"].str.strip()
df_historical_data["home"].head(3).tolist()

## 7. Séparer le score avec `str.split`

In [ ]:
df_historical_data[["HomeGoals", "AwayGoals"]] = df_historical_data["score"].str.split("–", expand=True)
df_historical_data = df_historical_data.drop(columns="score")
df_historical_data.head(3)

## 8. Renommer avec `rename` et convertir avec `astype`

Avec pandas 3, les colonnes de texte s'affichent `str` au lieu de `object`.

In [ ]:
df_historical_data = df_historical_data.rename(
    columns={"home": "HomeTeam", "away": "AwayTeam", "year": "Year"}
)
df_historical_data.dtypes

In [ ]:
df_historical_data = df_historical_data.astype({"HomeGoals": int, "AwayGoals": int})
df_historical_data.dtypes

## 9. Vérifier le résultat

Attendu : 900 matchs et 2548 buts.

In [ ]:
buts = df_historical_data["HomeGoals"].sum() + df_historical_data["AwayGoals"].sum()
print("Matchs :", len(df_historical_data), "| Buts :", buts)
df_historical_data.groupby("Year").size().to_dict()

## 10. Exporter avec `to_csv`

In [ ]:
df_historical_data.to_csv("clean_fifa_worldcup_matches.csv", index=False)
df_fixture.to_csv("clean_fifa_worldcup_fixture.csv", index=False)

## Tout le code

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/14-fonctions-pandas-pour-data-science/"

df_historical_data = pd.read_csv(BASE + "fifa_worldcup_matches.csv")
df_fixture = pd.read_csv(BASE + "fifa_worldcup_fixture.csv")
df_missing_data = pd.read_csv(BASE + "fifa_worldcup_missing_data.csv")

# 1. Nettoyer df_fixture
df_fixture["home"] = df_fixture["home"].str.strip()
df_fixture["away"] = df_fixture["away"].str.strip()

# 2. Supprimer les lignes vides de df_missing_data et fusionner
df_missing_data = df_missing_data.dropna()
df_historical_data = pd.concat([df_historical_data, df_missing_data], ignore_index=True)
df_historical_data = (
    df_historical_data
    .drop_duplicates()
    .sort_values("year", kind="stable")
    .reset_index(drop=True)
)

# 3. Supprimer le walkover Suède - Autriche (1938)
walkover = df_historical_data[
    df_historical_data["home"].str.contains("Sweden")
    & df_historical_data["away"].str.contains("Austria")
].index
df_historical_data = df_historical_data.drop(index=walkover)

# 4. Corriger les scores et les noms d'équipes
df_historical_data["score"] = df_historical_data["score"].str.replace(r"[^\d–]", "", regex=True)
df_historical_data["home"] = df_historical_data["home"].str.strip()
df_historical_data["away"] = df_historical_data["away"].str.strip()

# 5. Transformer : séparer le score, renommer, convertir
df_historical_data[["HomeGoals", "AwayGoals"]] = df_historical_data["score"].str.split("–", expand=True)
df_historical_data = df_historical_data.drop(columns="score")
df_historical_data = df_historical_data.rename(
    columns={"home": "HomeTeam", "away": "AwayTeam", "year": "Year"}
)
df_historical_data = df_historical_data.astype({"HomeGoals": int, "AwayGoals": int})

# 6. Exporter
df_historical_data.to_csv("clean_fifa_worldcup_matches.csv", index=False)
df_fixture.to_csv("clean_fifa_worldcup_fixture.csv", index=False)
df_historical_data.head()